In [ ]:
import math
from datetime import timedelta
from operator import attrgetter

import matplotlib.pyplot as plt
import numpy as np
import trajan as ta
import xarray as xr
import pandas as pd
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

import parcels
from parcels import StatusCode

In [ ]:
flowfield_env = xr.open_dataset("cmems_mod_glo_phy-curr.nc")
flowfield_env

In [ ]:
flowfield_env = flowfield_env.squeeze()
flowfield_env

In [ ]:
flowfield_env.uo.data[:,:,0]=0
flowfield_env.uo.data[:,:,-1]=0
flowfield_env.uo.data[:,0,:]=0
flowfield_env.uo.data[:,-1,:]=0

In [ ]:
flowfield_env.vo.data[:,:,0]=0
flowfield_env.vo.data[:,:,-1]=0
flowfield_env.vo.data[:,0,:]=0
flowfield_env.vo.data[:,-1,:]=0

In [ ]:
flowfield_env.uo.mean("time").plot()

In [ ]:
variables = {
    "U": "uo",
    "V": "vo",
}

In [ ]:
dimensions = {
    "lon": "longitude",
    "lat": "latitude",
    "time": "time"
}

In [ ]:
fieldset = parcels.FieldSet.from_xarray_dataset(flowfield_env, variables, dimensions)

In [ ]:
print(fieldset)

In [ ]:
fieldset.computeTimeChunk()

plt.pcolormesh(fieldset.U.grid.lon, fieldset.U.grid.lat, fieldset.U.data[9, :, :])
plt.xlabel("Zonal distance [m]")
plt.ylabel("Meridional distance [m]")
plt.colorbar()
plt.show()

In [ ]:
lon_grid = np.linspace(-25.15, -24.85, 100)
lat_grid = np.linspace(16.70, 17.00, 100)

In [ ]:
delta = 0.02

In [ ]:
lons = []
lats = []

for lon in lon_grid:
    for lat in lat_grid:

        lons.append(lon)
        lats.append(lat)

        lons.append(lon + delta)
        lats.append(lat)

        lons.append(lon)
        lats.append(lat + delta)

In [ ]:
pset = parcels.ParticleSet.from_list(
    fieldset=fieldset,
    pclass=parcels.JITParticle,
    lon=lons,
    lat=lats
)

In [ ]:
print(pset)

In [ ]:
plt.pcolormesh(fieldset.U.grid.lon, fieldset.U.grid.lat, fieldset.U.data[10, :, :])
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.colorbar()

plt.plot(pset.lon, pset.lat, "ko", markersize=1)
plt.show()

In [ ]:
output_file = pset.ParticleFile(
    name="CV_gridParticles.zarr",  # the file name
    outputdt=timedelta(hours=1),  # the time step of the outputs
)

print(output_file)

In [ ]:
# Recovery kernels
def CheckOutOfBounds(particle, fieldset, time):
    if particle.state == StatusCode.ErrorOutOfBounds:
        particle.delete()

def CheckError(particle, fieldset, time):
    if particle.state >= 50:
        particle.delete()

# Build kernel
kernels = (
    pset.Kernel(parcels.AdvectionRK4)
    + pset.Kernel(CheckOutOfBounds)
)

# Execute
pset.execute(
    kernels,
    runtime=timedelta(days=10),
    dt=timedelta(minutes=5),
    output_file=output_file,
)

In [ ]:
print(pset)

plt.pcolormesh(fieldset.U.grid.lon, fieldset.U.grid.lat, fieldset.U.data[0, :, :])
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.colorbar()

plt.plot(pset.lon, pset.lat, "ko", markersize = 1)
plt.show()

In [ ]:
final_lon = pset.lon
final_lon

In [ ]:
final_lat = pset.lat
final_lat

In [ ]:
center = []
xshift = []
yshift = []

for i in range(0, len(final_lon), 3):
    center.append([final_lon[i], final_lat[i]])
    xshift.append([final_lon[i+1], final_lat[i+1]])
    yshift.append([final_lon[i+2], final_lat[i+2]])

center = np.array(center)
xshift = np.array(xshift)
yshift = np.array(yshift)

In [ ]:
len(final_lon)

In [ ]:
len(final_lat)

In [ ]:
dXdx = (xshift[:,0] - center[:,0]) / delta
dYdx = (xshift[:,1] - center[:,1]) / delta

dXdy = (yshift[:,0] - center[:,0]) / delta
dYdy = (yshift[:,1] - center[:,1]) / delta

In [ ]:
dXdx.shape

In [ ]:
F = np.array([
    [dXdx, dXdy],
    [dYdx, dYdy]
])

In [ ]:
print(F)

In [ ]:
F.shape

In [ ]:
C = np.einsum("kil,kjl->ijl",F,F)
C

In [ ]:
C_T = np.transpose(C, (2, 0, 1))
C_T

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(C_T)

In [ ]:
eigenvalues

In [ ]:
eigenvectors

In [ ]:
np.argmax(eigenvalues,axis=1).mean()

In [ ]:
ev_max = np.where((np.argmax(eigenvalues,axis=1) == 0)[:,np.newaxis], eigenvectors[:,0,:], eigenvectors[:,1,:])
ev_max

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
for n in range(1000):
    _x = [lons[n] + delta * ev_max[n, 0], lons[n] - delta * ev_max[n, 0]]
    _y = [lats[n] + delta * ev_max[n, 1], lats[n] - delta * ev_max[n, 1]]
    ax.plot(_x, _y, 'r-', linewidth=0.5, alpha=1)

In [ ]:
max_eigenvalues = np.max(eigenvalues,axis=1)
max_eigenvalues

In [ ]:
plt.hist(max_eigenvalues)

In [ ]:
pd.Series(max_eigenvalues).quantile(np.linspace(0,1,31))

In [ ]:
T = 10*24*60*60
T

In [ ]:
FTLE = (1/(2*T)) * np.log(max_eigenvalues)
FTLE

In [ ]:
FTLE.shape

In [ ]:
plt.hist(FTLE*3600*24)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 12))

ax.scatter(lons[0::3], lats[0::3], 10, FTLE*24*3600)

for n in range(0,10000,1):
    x = [lons[n] + delta * ev_max[n, 0], lons[n] - delta * ev_max[n, 0]]
    _y = [lats[n] + delta * ev_max[n, 1], lats[n] - delta * ev_max[n, 1]]
    ax.plot(_x, _y, 'r-', linewidth=0.5, alpha=1)
    
fig.set_dpi(200)
plt.show()